### My little KNN
using hog with knn to detect the numbers with accuracy of 0.9956


In [33]:
from pathlib import Path
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from scipy.ndimage import center_of_mass, shift
from scipy.ndimage import binary_closing


In [34]:
TrainingDirectory = Path("iivp-2026-challenge/train/train")


In [35]:
X_train, y_train = [], []
for class_dir in sorted(TrainingDirectory.iterdir(), key=lambda p: int(p.name)):
    label = int(class_dir.name)
    for png in class_dir.glob("*.png"):
        X_train.append(np.array(Image.open(png), dtype=np.uint8))
        y_train.append(label)

X_train = np.stack(X_train)
y_train = np.array(y_train)

print(X_train.shape, y_train.shape)

(17000, 32, 32) (17000,)


In [36]:
def recenter(img):
    cy,cx = center_of_mass(img)
    if np.isnan(cy):
        return img
    return shift(img, (img.shape[0]/2 - cy, img.shape[1]/2 -cx), order=1)

X_train_centered = np.array([recenter(img) for img in X_train])


In [37]:
def extractH(img):
    return hog(
        img, orientations=9, pixels_per_cell=(4,4), cells_per_block=(2,2), block_norm='L2-Hys',)


print(extractH(X_train[0]).shape)
X_hog = np.array([extractH(img) for img in X_train])
print(X_hog.shape)

(1764,)
(17000, 1764)


In [38]:
X_tr_hog, X_val_hog, y_tr, y_val = train_test_split(
    X_hog, y_train, test_size=0.2, stratify=y_train, random_state=42
)

print(X_tr_hog.shape, X_val_hog.shape)


(13600, 1764) (3400, 1764)


In [42]:
knn = KNeighborsClassifier(3,n_jobs=-1)
knn.fit(X_tr_hog, y_tr)

y_pred = knn.predict(X_val_hog)
print(f"accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(classification_report(y_val, y_pred))


accuracy: 0.9956
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       340
           1       1.00      1.00      1.00       340
           2       0.97      1.00      0.99       340
           3       1.00      0.98      0.99       340
           4       1.00      1.00      1.00       340
           5       1.00      0.99      0.99       340
           6       1.00      0.99      1.00       340
           7       1.00      1.00      1.00       340
           8       0.99      1.00      1.00       340
           9       0.99      1.00      1.00       340

    accuracy                           1.00      3400
   macro avg       1.00      1.00      1.00      3400
weighted avg       1.00      1.00      1.00      3400

